# Generate cycle-level cardiotoxicity modelling table

This notebook is adjusted for your repository structure:

```text
repo_root/
├── data_outputs/
├── exploration_sql_files/
│   ├── cohort_outcome_sql/
│   ├── diagnoses_sql/
│   ├── drug_cycles_sql/
│   ├── echo_sql/
│   ├── prescriptions_sql/
│   └── procedure_sql/
├── mimic-iv-3.1/
├── mimic-iv-ecg/
├── mimic-iv-echo/
└── generate_cycle_modeling_table.ipynb
```

The notebook creates one modelling row per patient cycle-like oncology exposure. You can later split in pandas by `subject_id`.

In [1]:
# If needed in your notebook environment:
# %pip install duckdb pandas pyarrow

from pathlib import Path
import os

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

## 1. Configure repository paths

If this notebook is located at your repository root, leave `REPO_ROOT = Path('.')`.

If you run it from somewhere else, set `REPO_ROOT` manually to your repo folder.

In [2]:
REPO_ROOT = Path(".").resolve()

SQL_ROOT = REPO_ROOT / "exploration_sql_files"
DIAGNOSES_SQL_DIR = SQL_ROOT / "diagnoses_sql"
PRESCRIPTIONS_SQL_DIR = SQL_ROOT / "prescriptions_sql"
DRUG_CYCLES_SQL_DIR = SQL_ROOT / "drug_cycles_sql"

MIMIC_HOSP_DIR = REPO_ROOT / "mimic-iv-3.1" / "hosp"
MIMIC_ECHO_DIR = REPO_ROOT / "mimic-iv-echo"

OUTPUT_DIR = REPO_ROOT / "data_outputs" / "cycle_modeling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("DIAGNOSES_SQL_DIR:", DIAGNOSES_SQL_DIR)
print("PRESCRIPTIONS_SQL_DIR:", PRESCRIPTIONS_SQL_DIR)
print("DRUG_CYCLES_SQL_DIR:", DRUG_CYCLES_SQL_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MIMIC_HOSP_DIR exists:", MIMIC_HOSP_DIR.exists())
print("MIMIC_ECHO_DIR exists:", MIMIC_ECHO_DIR.exists())

REPO_ROOT: /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection
DIAGNOSES_SQL_DIR: /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/diagnoses_sql
PRESCRIPTIONS_SQL_DIR: /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/prescriptions_sql
DRUG_CYCLES_SQL_DIR: /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/drug_cycles_sql
OUTPUT_DIR: /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/data_outputs/cycle_modeling
MIMIC_HOSP_DIR exists: True
MIMIC_ECHO_DIR exists: True


## 2. SQL files expected by this notebook

The base SQL comes from your existing folders. The new cycle-labelling SQL should live under:

```text
exploration_sql_files/drug_cycles_sql/
```

In [3]:
BASE_SQL_PATHS = [
    DIAGNOSES_SQL_DIR / "active_cancer.sql",
    DIAGNOSES_SQL_DIR / "personal_history_cancer.sql",
    DIAGNOSES_SQL_DIR / "history_and_active.sql",
    PRESCRIPTIONS_SQL_DIR / "prescriptions_count_regex.sql",
]

CYCLE_SQL_PATHS = [
    DRUG_CYCLES_SQL_DIR / "00_parameters_and_windows.sql",
    DRUG_CYCLES_SQL_DIR / "01_drug_classification_and_first_drug.sql",
    DRUG_CYCLES_SQL_DIR / "02_cycle_exposures.sql",
    DRUG_CYCLES_SQL_DIR / "03_lvef_toxicity_events.sql",
    DRUG_CYCLES_SQL_DIR / "04_cv_toxicity_events.sql",
    DRUG_CYCLES_SQL_DIR / "05_first_toxicity_and_observation.sql",
    DRUG_CYCLES_SQL_DIR / "06_final_modeling_table.sql",
]

for path in BASE_SQL_PATHS + CYCLE_SQL_PATHS:
    print("OK" if path.exists() else "MISSING", path)

OK /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/diagnoses_sql/active_cancer.sql
OK /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/diagnoses_sql/personal_history_cancer.sql
OK /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/diagnoses_sql/history_and_active.sql
OK /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/prescriptions_sql/prescriptions_count_regex.sql
OK /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/drug_cycles_sql/00_parameters_and_windows.sql
OK /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/drug_cycles_sql/01_drug_classification_and_first_drug.sql
OK /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_

## 3. Helper functions

In [4]:
def execute_sql_file(con: duckdb.DuckDBPyConnection, path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing SQL file: {path}")

    sql = path.read_text()

    # Make your exploration SQL rerunnable in notebook sessions.
    sql = sql.replace("CREATE VIEW active_cancer", "CREATE OR REPLACE VIEW active_cancer")
    sql = sql.replace("CREATE VIEW oncology_drugs", "CREATE OR REPLACE VIEW oncology_drugs")

    con.execute(sql)


def execute_sql_files(con: duckdb.DuckDBPyConnection, paths: list[Path]) -> None:
    for path in paths:
        print(f"running {path.relative_to(REPO_ROOT) if path.is_relative_to(REPO_ROOT) else path}")
        execute_sql_file(con, path)


def table_exists(con: duckdb.DuckDBPyConnection, name: str) -> bool:
    result = con.execute(
        """
        SELECT COUNT(*) AS n
        FROM information_schema.tables
        WHERE table_name = ?
        """,
        [name],
    ).fetchone()
    return bool(result and result[0] > 0)


def count_rows(con: duckdb.DuckDBPyConnection, name: str) -> int:
    return con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]


def preview(con: duckdb.DuckDBPyConnection, name: str, n: int = 5) -> pd.DataFrame:
    return con.execute(f"SELECT * FROM {name} LIMIT {n}").df()


def describe_view(con: duckdb.DuckDBPyConnection, name: str) -> pd.DataFrame:
    return con.execute(f"DESCRIBE {name}").df()


def write_dataframe(df: pd.DataFrame, output_dir: Path, stem: str) -> None:
    csv_path = output_dir / f"{stem}.csv"
    df.to_csv(csv_path, index=False)
    print(f"wrote {csv_path}")

    parquet_path = output_dir / f"{stem}.parquet"
    try:
        df.to_parquet(parquet_path, index=False)
        print(f"wrote {parquet_path}")
    except Exception as exc:
        print(f"skipped parquet for {stem}: {exc}")

## 4. Connect to DuckDB

The SQL files use relative paths like:

```text
mimic-iv-3.1/hosp/prescriptions.csv
mimic-iv-echo/structured-measurement.csv
```

So the notebook changes the working directory to `REPO_ROOT` before executing SQL.

In [5]:
os.chdir(REPO_ROOT)

DB_TARGET = ":memory:"
# Optional persistent DB:
# DB_TARGET = str(REPO_ROOT / "data_outputs" / "cycle_modeling.duckdb")

con = duckdb.connect(DB_TARGET)
print("Connected to", DB_TARGET)
print("Current working directory:", Path.cwd())

Connected to :memory:
Current working directory: /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection


## 5. Load base cohort + oncology prescriptions

In [6]:
execute_sql_files(con, BASE_SQL_PATHS)

print("all_cancer_patients exists:", table_exists(con, "all_cancer_patients"))
print("oncology_drugs exists:", table_exists(con, "oncology_drugs"))
print("all_cancer_patients rows:", f"{count_rows(con, 'all_cancer_patients'):,}")
print("oncology_drugs rows:", f"{count_rows(con, 'oncology_drugs'):,}")

running exploration_sql_files/diagnoses_sql/active_cancer.sql
running exploration_sql_files/diagnoses_sql/personal_history_cancer.sql
running exploration_sql_files/diagnoses_sql/history_and_active.sql
running exploration_sql_files/prescriptions_sql/prescriptions_count_regex.sql
all_cancer_patients exists: True
oncology_drugs exists: True
all_cancer_patients rows: 48,477
oncology_drugs rows: 7,743


In [7]:
preview(con, "oncology_drugs", 10)

,subject_id,hadm_id,pharmacy_id,starttime,stoptime,drug
0,10003019,20277210,25458103,2175-11-14 16:00:00,2175-11-15 15:00:00,doxorubicin
1,10003019,22774359,33098165,2175-10-01 17:00:00,2175-10-16 16:00:00,doxorubicin
2,10003400,26090619,59864436,2134-06-06 16:00:00,2134-06-07 19:00:00,lenalidomide (revlimide)15mg
3,10010231,25499227,80302652,2117-11-10 19:00:00,2117-11-13 18:00:00,daunorubicin
4,10012768,20255893,18062841,2111-08-18 15:00:00,2111-08-21 19:00:00,bortezomib
5,10012768,20255893,72049952,2111-08-13 15:00:00,2111-08-18 16:00:00,bortezomib
6,10012768,20255893,94657909,2111-08-13 22:00:00,2111-08-18 14:00:00,bortezomib
7,10016991,27389040,8836777,2124-09-16 12:00:00,2124-09-16 23:00:00,fluorouracil
8,10016991,27389040,12802537,2124-09-16 11:00:00,2124-09-16 23:00:00,fluorouracil
9,10022373,27450651,56873260,2150-05-21 18:00:00,2150-06-04 08:00:00,fluorouracil


## 6. Run cycle-modelling SQL interactively

You can run these cells one by one and inspect each intermediate view.

### 6.1 Parameters and drug-specific windows

In [8]:
execute_sql_file(con, DRUG_CYCLES_SQL_DIR / "00_parameters_and_windows.sql")
preview(con, "drug_toxicity_windows", 20)

,drug_class,toxicity_window_days,window_rationale
0,anthracycline,365,Early-onset CTRCD/HF commonly assessed within 1 year
1,immune_checkpoint_inhibitor,90,"ICI myocarditis usually occurs early, often within weeks to 3 months"
2,her2_targeted,365,HER2-related LV dysfunction commonly monitored over months to 1 year
3,taxane,90,Shorter window for acute/subacute CV events
4,fluoropyrimidine,30,Often acute/subacute ischemia/vasospasm window
5,vegf_inhibitor,180,Hypertension/HF/ischemic risk over months
6,egfr_inhibitor,180,General CV surveillance window
7,tyrosine_kinase_inhibitor,180,General CV surveillance window
8,proteasome_inhibitor,180,HF/ischemia/arrhythmia risk over months
9,immunomodulatory_agent,180,Thrombotic/CV risk over months


### 6.2 Drug classification and first drug anchor

In [9]:
execute_sql_file(con, DRUG_CYCLES_SQL_DIR / "01_drug_classification_and_first_drug.sql")

print("oncology_drugs_classified rows:", f"{count_rows(con, 'oncology_drugs_classified'):,}")
print("cancer_first_drug rows:", f"{count_rows(con, 'cancer_first_drug'):,}")

preview(con, "oncology_drugs_classified", 10)

oncology_drugs_classified rows: 7,743
cancer_first_drug rows: 2,545


,subject_id,hadm_id,pharmacy_id,starttime,stoptime,drug,drug_class
0,10003019,20277210,25458103,2175-11-14 16:00:00,2175-11-15 15:00:00,doxorubicin,anthracycline
1,10003019,22774359,33098165,2175-10-01 17:00:00,2175-10-16 16:00:00,doxorubicin,anthracycline
2,10003400,26090619,59864436,2134-06-06 16:00:00,2134-06-07 19:00:00,lenalidomide (revlimide)15mg,immunomodulatory_agent
3,10010231,25499227,80302652,2117-11-10 19:00:00,2117-11-13 18:00:00,daunorubicin,anthracycline
4,10012768,20255893,18062841,2111-08-18 15:00:00,2111-08-21 19:00:00,bortezomib,proteasome_inhibitor
5,10012768,20255893,72049952,2111-08-13 15:00:00,2111-08-18 16:00:00,bortezomib,proteasome_inhibitor
6,10012768,20255893,94657909,2111-08-13 22:00:00,2111-08-18 14:00:00,bortezomib,proteasome_inhibitor
7,10016991,27389040,8836777,2124-09-16 12:00:00,2124-09-16 23:00:00,fluorouracil,fluoropyrimidine
8,10016991,27389040,12802537,2124-09-16 11:00:00,2124-09-16 23:00:00,fluorouracil,fluoropyrimidine
9,10022373,27450651,56873260,2150-05-21 18:00:00,2150-06-04 08:00:00,fluorouracil,fluoropyrimidine


In [10]:
con.execute("""
SELECT
    drug_class,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT subject_id) AS n_patients
FROM oncology_drugs_classified
GROUP BY drug_class
ORDER BY n_rows DESC
""").df()

,drug_class,n_rows,n_patients
0,anthracycline,3791,1231
1,taxane,1233,502
2,fluoropyrimidine,1067,418
3,proteasome_inhibitor,752,315
4,tyrosine_kinase_inhibitor,395,122
5,immunomodulatory_agent,197,111
6,vegf_inhibitor,101,75
7,her2_targeted,97,50
8,egfr_inhibitor,64,18
9,immune_checkpoint_inhibitor,46,29


### 6.3 Cycle-like exposures

This collapses nearby oncology drug starts into one cycle-like row. Change `cycle_gap_days` in `00_parameters_and_windows.sql` if you want a wider/narrower grouping rule.

In [11]:
execute_sql_file(con, DRUG_CYCLES_SQL_DIR / "02_cycle_exposures.sql")

print("oncology_cycle_exposures rows:", f"{count_rows(con, 'oncology_cycle_exposures'):,}")
preview(con, "oncology_cycle_exposures", 10)

oncology_cycle_exposures rows: 5,420


,subject_id,cycle_number,prediction_time,cycle_start_date,cycle_end_date,n_exposure_start_days_in_cycle,n_prescription_rows_in_cycle,drugs_in_cycle,drug_classes_in_cycle,toxicity_window_days,window_rationales,exposed_anthracycline,exposed_immune_checkpoint_inhibitor,exposed_her2_targeted,exposed_taxane,exposed_fluoropyrimidine,exposed_vegf_inhibitor,exposed_egfr_inhibitor,exposed_tyrosine_kinase_inhibitor,exposed_proteasome_inhibitor,exposed_immunomodulatory_agent
0,10150465,1.0,2155-07-14 20:00:00,2155-07-14,2155-07-14,1,1.0,paclitaxel (taxol),taxane,90,Shorter window for acute/subacute CV events,0,0,0,1,0,0,0,0,0,0
1,10381484,2.0,2175-10-24 17:00:00,2175-10-24,2175-10-24,2,4.0,fluorouracil,fluoropyrimidine,30,Often acute/subacute ischemia/vasospasm window,0,0,0,0,1,0,0,0,0,0
2,10386562,2.0,2169-06-14 19:00:00,2169-06-14,2169-06-14,1,1.0,imatinib mesylate,tyrosine_kinase_inhibitor,180,General CV surveillance window,0,0,0,0,0,0,0,1,0,0
3,10617538,5.0,2160-03-06 22:00:00,2160-03-06,2160-03-06,1,1.0,doxorubicin,anthracycline,365,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0
4,10617964,3.0,2117-12-08 12:00:00,2117-12-08,2117-12-08,1,1.0,docetaxel (taxotere),taxane,90,Shorter window for acute/subacute CV events,0,0,0,1,0,0,0,0,0,0
5,10623751,1.0,2181-05-09 12:00:00,2181-05-09,2181-05-09,2,4.0,bortezomib,proteasome_inhibitor,180,HF/ischemia/arrhythmia risk over months,0,0,0,0,0,0,0,0,1,0
6,10640241,4.0,2168-10-21 09:00:00,2168-10-21,2168-10-21,1,1.0,liposomal doxorubicin (doxil),anthracycline,365,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0
7,10826901,4.0,2135-08-05 17:00:00,2135-08-05,2135-08-05,1,1.0,doxorubicin,anthracycline,365,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0
8,10974928,5.0,2169-01-30 12:00:00,2169-01-30,2169-01-30,1,1.0,bevacizumab (avastin),vegf_inhibitor,180,Hypertension/HF/ischemic risk over months,0,0,0,0,0,1,0,0,0,0
9,10976101,2.0,2170-08-15 20:00:00,2170-08-15,2170-08-15,1,1.0,fluorouracil,fluoropyrimidine,30,Often acute/subacute ischemia/vasospasm window,0,0,0,0,1,0,0,0,0,0


In [12]:
con.execute("""
SELECT
    cycle_number,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT subject_id) AS n_patients
FROM oncology_cycle_exposures
GROUP BY cycle_number
ORDER BY cycle_number
LIMIT 25
""").df()

,cycle_number,n_rows,n_patients
0,1.0,2545,2545
1,2.0,1057,1057
2,3.0,621,621
3,4.0,472,472
4,5.0,347,347
5,6.0,261,261
6,7.0,47,47
7,8.0,30,30
8,9.0,17,17
9,10.0,6,6


### 6.4 LVEF toxicity events

In [13]:
execute_sql_file(con, DRUG_CYCLES_SQL_DIR / "03_lvef_toxicity_events.sql")

print("all_lvef rows:", f"{count_rows(con, 'all_lvef'):,}")
print("baseline_lvef_pre_first_drug rows:", f"{count_rows(con, 'baseline_lvef_pre_first_drug'):,}")
print("lvef_toxicity_events rows:", f"{count_rows(con, 'lvef_toxicity_events'):,}")

preview(con, "lvef_toxicity_events", 10)

all_lvef rows: 147,794
baseline_lvef_pre_first_drug rows: 2,545
lvef_toxicity_events rows: 338


,subject_id,toxicity_time,toxicity_type,baseline_time,baseline_lvef,event_lvef,absolute_lvef_drop
0,10354409,2134-08-16 10:00:00,lvef_ctrctd,2134-03-30 09:58:00,45.0,30.0,15.0
1,10533554,2176-03-25 13:00:00,lvef_ctrctd,2171-07-07 13:00:00,60.0,45.0,15.0
2,10533554,2177-01-22 11:05:00,lvef_ctrctd,2171-07-07 13:00:00,60.0,45.0,15.0
3,10533554,2177-01-31 01:08:00,lvef_ctrctd,2171-07-07 13:00:00,60.0,40.0,20.0
4,10533554,2177-03-09 09:24:00,lvef_ctrctd,2171-07-07 13:00:00,60.0,45.0,15.0
5,10533554,2177-12-08 10:59:00,lvef_ctrctd,2171-07-07 13:00:00,60.0,45.0,15.0
6,10533554,2179-03-27 11:58:00,lvef_ctrctd,2171-07-07 13:00:00,60.0,40.0,20.0
7,10533554,2180-09-12 14:00:00,lvef_ctrctd,2171-07-07 13:00:00,60.0,40.0,20.0
8,10533554,2181-07-08 14:21:00,lvef_ctrctd,2171-07-07 13:00:00,60.0,45.0,15.0
9,10534626,2183-11-14 00:00:00,lvef_ctrctd,2183-06-01 10:27:00,67.0,40.0,27.0


### 6.5 ICD/admission cardiovascular toxicity events

In [14]:
execute_sql_file(con, DRUG_CYCLES_SQL_DIR / "04_cv_toxicity_events.sql")

print("cancer_cv_admission_events rows:", f"{count_rows(con, 'cancer_cv_admission_events'):,}")
print("cv_toxicity_events rows:", f"{count_rows(con, 'cv_toxicity_events'):,}")
print("pre_existing_cv_history rows:", f"{count_rows(con, 'pre_existing_cv_history'):,}")

preview(con, "cv_toxicity_events", 10)

cancer_cv_admission_events rows: 17,962
cv_toxicity_events rows: 2,827
pre_existing_cv_history rows: 813


,subject_id,toxicity_time,toxicity_type,hadm_id,cv_event_icd_codes
0,13198542,2148-01-15 16:39:00,cv_diagnosis_admission,20979419,D473 | D89811 | E872 | G4440 | I110 | I5031 | J9622 | L0889 | R008 | T361X5A | T865 | Z856 | Z86718
1,13198542,2147-09-22 18:32:00,cv_diagnosis_admission,24317523,C9101 | D509 | D72829 | D801 | D89812 | E872 | I110 | I5030 | J42 | J9622 | L859 | R7989 | S91001A | T865 | X58XXXA ...
2,15553257,2188-06-03 15:42:00,cv_diagnosis_admission,22122732,D472 | D62 | E7800 | F0390 | H409 | I110 | I2720 | I4892 | I5032 | M4304 | N400 | R791 | S50312A | S72142A | T8141XA...
3,13085886,2159-06-20 17:41:00,cv_diagnosis_admission,23253808,B009 | C92Z2 | D1802 | D6481 | D6959 | D701 | D89813 | E43 | E785 | I472 | I951 | L309 | N179 | R197 | R5081 | R51 |...
4,13128939,2138-05-14 14:13:00,cv_diagnosis_admission,22569545,D61818 | F1011 | F1111 | F1211 | I110 | I509 | K219 | K429 | K529 | M549 | R1084 | R51 | R945 | Z20828 | Z8505 | Z944
5,13160692,2126-12-07 18:02:00,cv_diagnosis_admission,22968951,C182 | D649 | D696 | E669 | E785 | H02402 | H409 | I272 | I4510 | K219 | Z6832 | Z8572 | Z86718 | Z8673 | Z9484
6,13278241,2146-04-23 13:22:00,cv_diagnosis_admission,22787023,B952 | B961 | C9000 | D630 | D631 | E1121 | E1122 | E1140 | E1152 | E11621 | E11649 | E1169 | E785 | F05 | I132 | I2...
7,13278241,2146-04-01 17:34:00,cv_diagnosis_admission,23069378,C9000 | D631 | D72829 | E1122 | E1140 | G8918 | I132 | I2510 | I4891 | I5030 | I739 | M85872 | N186 | R627 | Z7901 |...
8,13312271,2172-11-06 12:39:00,cv_diagnosis_admission,22067857,20300 | 25080 | 2724 | 40390 | 41400 | 53081 | 5859 | V1254 | V422 | V4581 | V5867
9,13336663,2131-02-18 17:41:00,cv_diagnosis_admission,20584894,042 | 20020 | 25000 | 2724 | 28749 | 32723 | 42789 | 4321 | 53081 | 7840 | 78701 | V4986 | V5865 | V5867 | V667 | V8741


### 6.6 First toxicity and observation evidence

In [15]:
execute_sql_file(con, DRUG_CYCLES_SQL_DIR / "05_first_toxicity_and_observation.sql")

print("all_cardiotoxicity_events rows:", f"{count_rows(con, 'all_cardiotoxicity_events'):,}")
print("first_cardiotoxicity_event rows:", f"{count_rows(con, 'first_cardiotoxicity_event'):,}")
print("patient_last_observation rows:", f"{count_rows(con, 'patient_last_observation'):,}")

preview(con, "first_cardiotoxicity_event", 10)

all_cardiotoxicity_events rows: 3,165
first_cardiotoxicity_event rows: 867
patient_last_observation rows: 229,103


,subject_id,first_toxicity_time,first_toxicity_type
0,11269230,2150-08-11 16:07:00,cv_diagnosis_admission
1,13607095,2149-02-26 14:57:00,lvef_ctrctd
2,17607781,2170-02-14 19:54:00,cv_diagnosis_admission
3,17624614,2160-08-15 02:08:00,cv_diagnosis_admission
4,18513809,2153-12-25 18:43:00,cv_diagnosis_admission
5,12279787,2137-06-08 20:43:00,cv_diagnosis_admission
6,12878207,2130-02-10 12:08:00,cv_diagnosis_admission
7,14359166,2147-02-20 19:23:00,cv_diagnosis_admission
8,17460070,2189-09-27 03:49:00,cv_diagnosis_admission
9,17587395,2129-05-21 15:05:00,cv_diagnosis_admission


### 6.7 Final modelling table

In [16]:
execute_sql_file(con, DRUG_CYCLES_SQL_DIR / "06_final_modeling_table.sql")

print("final_cycle_modeling_table rows:", f"{count_rows(con, 'final_cycle_modeling_table'):,}")
print("final_cycle_binary_modeling_table rows:", f"{count_rows(con, 'final_cycle_binary_modeling_table'):,}")

preview(con, "final_cycle_modeling_table", 10)

final_cycle_modeling_table rows: 5,420
final_cycle_binary_modeling_table rows: 2,555


,subject_id,cycle_number,prediction_time,cycle_start_date,cycle_end_date,n_exposure_start_days_in_cycle,n_prescription_rows_in_cycle,drugs_in_cycle,drug_classes_in_cycle,toxicity_window_days,prediction_window_end,window_rationales,exposed_anthracycline,exposed_immune_checkpoint_inhibitor,exposed_her2_targeted,exposed_taxane,exposed_fluoropyrimidine,exposed_vegf_inhibitor,exposed_egfr_inhibitor,exposed_tyrosine_kinase_inhibitor,exposed_proteasome_inhibitor,exposed_immunomodulatory_agent,first_toxicity_time,first_toxicity_type,toxicity_in_window,pre_existing_cv_history,baseline_time,baseline_lvef,last_observation_time,observed_through_prediction_window,has_observation_in_prediction_window,eligible_for_binary_label,binary_label,label
0,11276023,1.0,2124-08-25 15:00:00,2124-08-25,2124-08-25,1,1.0,fluorouracil,fluoropyrimidine,30,2124-09-24 15:00:00,Often acute/subacute ischemia/vasospasm window,0,0,0,0,1,0,0,0,0,0,2124-08-29 22:51:00,cv_diagnosis_admission,1,1,NaT,NaN,2124-09-27 19:05:00,1,1,1,1,positive
1,10811360,1.0,2150-09-13 14:00:00,2150-09-13,2150-09-13,1,1.0,doxorubicin,anthracycline,365,2151-09-13 14:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2150-09-27 14:42:00,cv_diagnosis_admission,1,1,NaT,NaN,2150-10-01 17:29:00,0,1,1,1,positive
2,10898223,6.0,2145-11-21 12:00:00,2145-11-21,2145-11-21,2,4.0,fluorouracil,fluoropyrimidine,30,2145-12-21 12:00:00,Often acute/subacute ischemia/vasospasm window,0,0,0,0,1,0,0,0,0,0,2145-11-07 08:04:00,cv_diagnosis_admission,0,1,2145-04-17 10:31:00,60.0,2145-12-19 15:00:00,0,1,0,<NA>,exclude_already_toxic
3,11027112,3.0,2206-12-07 16:00:00,2206-12-07,2206-12-07,2,4.0,doxorubicin,anthracycline,365,2207-12-07 16:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2206-11-18 08:38:00,cv_diagnosis_admission,0,1,2206-10-28 16:11:00,55.0,2207-10-26 10:24:00,0,1,0,<NA>,exclude_already_toxic
4,13839166,4.0,2121-06-23 21:00:00,2121-06-23,2121-06-23,1,1.0,doxorubicin,anthracycline,365,2122-06-23 21:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2122-07-06 19:58:00,cv_diagnosis_admission,0,1,2121-04-01 00:00:00,75.0,2122-07-06 19:58:00,1,1,1,0,negative_observed
5,18642101,2.0,2181-11-23 15:00:00,2181-11-23,2181-11-23,1,1.0,doxorubicin,anthracycline,365,2182-11-23 15:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2181-12-23 14:44:00,cv_diagnosis_admission,1,1,2181-10-31 14:01:00,60.0,2188-06-01 08:00:00,1,1,1,1,positive
6,14110681,1.0,2138-09-27 09:00:00,2138-09-27,2138-09-27,1,1.0,paclitaxel (taxol),taxane,90,2138-12-26 09:00:00,Shorter window for acute/subacute CV events,0,0,0,1,0,0,0,0,0,0,2138-10-31 18:04:00,cv_diagnosis_admission,1,1,NaT,NaN,2139-05-04 21:14:00,1,1,1,1,positive
7,14282289,1.0,2143-07-16 10:00:00,2143-07-16,2143-07-16,1,1.0,paclitaxel (taxol),taxane,90,2143-10-14 10:00:00,Shorter window for acute/subacute CV events,0,0,0,1,0,0,0,0,0,0,2144-04-21 05:38:00,cv_diagnosis_admission,0,1,2143-03-09 10:31:00,60.0,2145-03-24 19:18:00,1,0,1,0,negative_observed
8,17521593,1.0,2144-03-21 13:00:00,2144-03-21,2144-03-21,1,1.0,doxorubicin,anthracycline,365,2145-03-21 13:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2144-05-12 19:55:00,cv_diagnosis_admission,1,1,2144-03-12 13:00:00,55.0,2144-05-30 10:00:00,0,1,1,1,positive
9,12397814,1.0,2144-04-12 12:00:00,2144-04-12,2144-04-14,5,17.0,docetaxel (taxotere) | fluorouracil,fluoropyrimidine | taxane,90,2144-07-11 12:00:00,Often acute/subacute ischemia/vasospasm window | Shorter window for acute/subacute CV events,0,0,0,1,1,0,0,0,0,0,2144-05-22 16:37:00,cv_diagnosis_admission,1,1,NaT,NaN,2144-05-22 16:37:00,0,1,1,1,positive


## 7. Load final views into pandas

In [17]:
final_df = con.execute("""
SELECT *
FROM final_cycle_modeling_table
ORDER BY subject_id, cycle_number
""").df()

binary_df = con.execute("""
SELECT *
FROM final_cycle_binary_modeling_table
ORDER BY subject_id, cycle_number
""").df()

final_df.shape, binary_df.shape

((5420, 34), (2555, 34))

In [18]:
final_df.head(10)

,subject_id,cycle_number,prediction_time,cycle_start_date,cycle_end_date,n_exposure_start_days_in_cycle,n_prescription_rows_in_cycle,drugs_in_cycle,drug_classes_in_cycle,toxicity_window_days,prediction_window_end,window_rationales,exposed_anthracycline,exposed_immune_checkpoint_inhibitor,exposed_her2_targeted,exposed_taxane,exposed_fluoropyrimidine,exposed_vegf_inhibitor,exposed_egfr_inhibitor,exposed_tyrosine_kinase_inhibitor,exposed_proteasome_inhibitor,exposed_immunomodulatory_agent,first_toxicity_time,first_toxicity_type,toxicity_in_window,pre_existing_cv_history,baseline_time,baseline_lvef,last_observation_time,observed_through_prediction_window,has_observation_in_prediction_window,eligible_for_binary_label,binary_label,label
0,10003019,1.0,2175-10-01 17:00:00,2175-10-01,2175-10-01,1,1.0,doxorubicin,anthracycline,365,2176-09-30 17:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2175-10-08 13:56:00,cv_diagnosis_admission,1,1,2175-09-25 17:00:00,60.0,2184-03-14 13:57:00,1,1,1,1,positive
1,10003019,2.0,2175-11-14 16:00:00,2175-11-14,2175-11-14,1,1.0,doxorubicin,anthracycline,365,2176-11-13 16:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2175-10-08 13:56:00,cv_diagnosis_admission,0,1,2175-09-25 17:00:00,60.0,2184-03-14 13:57:00,1,1,0,<NA>,exclude_already_toxic
2,10003400,1.0,2134-06-06 16:00:00,2134-06-06,2134-06-06,1,1.0,lenalidomide (revlimide)15mg,immunomodulatory_agent,180,2134-12-03 16:00:00,Thrombotic/CV risk over months,0,0,0,0,0,0,0,0,0,1,2136-11-04 20:43:00,cv_diagnosis_admission,0,1,2134-04-04 14:00:00,55.0,2137-08-05 14:31:00,1,0,1,0,negative_observed
3,10010231,1.0,2117-11-10 19:00:00,2117-11-10,2117-11-10,1,1.0,daunorubicin,anthracycline,365,2118-11-10 19:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,NaT,NaN,0,0,2117-11-09 09:00:00,60.0,2118-04-17 11:03:00,0,1,0,<NA>,unknown_no_followup_evidence
4,10012768,1.0,2111-08-13 15:00:00,2111-08-13,2111-08-18,3,5.0,bortezomib,proteasome_inhibitor,180,2112-02-09 15:00:00,HF/ischemia/arrhythmia risk over months,0,0,0,0,0,0,0,0,1,0,NaT,NaN,0,0,NaT,NaN,2112-01-06 15:01:00,0,1,0,<NA>,unknown_no_followup_evidence
5,10016991,1.0,2124-09-16 11:00:00,2124-09-16,2124-09-16,2,4.0,fluorouracil,fluoropyrimidine,30,2124-10-16 11:00:00,Often acute/subacute ischemia/vasospasm window,0,0,0,0,1,0,0,0,0,0,NaT,NaN,0,0,NaT,NaN,2124-09-16 12:00:00,0,1,0,<NA>,unknown_no_followup_evidence
6,10022373,1.0,2150-05-21 18:00:00,2150-05-21,2150-05-21,2,4.0,fluorouracil,fluoropyrimidine,30,2150-06-20 18:00:00,Often acute/subacute ischemia/vasospasm window,0,0,0,0,1,0,0,0,0,0,NaT,NaN,0,0,NaT,NaN,2150-07-19 00:26:00,1,0,1,0,negative_observed
7,10027393,1.0,2128-10-30 18:00:00,2128-10-30,2128-10-30,2,4.0,fluorouracil,fluoropyrimidine,30,2128-11-29 18:00:00,Often acute/subacute ischemia/vasospasm window,0,0,0,0,1,0,0,0,0,0,NaT,NaN,0,0,NaT,NaN,2128-10-30 18:00:00,0,0,0,<NA>,unknown_no_followup_evidence
8,10030549,1.0,2141-08-16 10:00:00,2141-08-16,2141-08-16,2,4.0,paclitaxel (taxol),taxane,90,2141-11-14 10:00:00,Shorter window for acute/subacute CV events,0,0,0,1,0,0,0,0,0,0,NaT,NaN,0,0,NaT,NaN,2141-11-12 01:45:00,0,1,0,<NA>,unknown_no_followup_evidence
9,10030549,2.0,2141-09-05 14:00:00,2141-09-05,2141-09-05,1,1.0,paclitaxel (taxol),taxane,90,2141-12-04 14:00:00,Shorter window for acute/subacute CV events,0,0,0,1,0,0,0,0,0,0,NaT,NaN,0,0,NaT,NaN,2141-11-12 01:45:00,0,1,0,<NA>,unknown_no_followup_evidence


## 8. Explore label quality

In [19]:
label_breakdown = (
    final_df.groupby("label", dropna=False)
    .agg(
        n_rows=("subject_id", "size"),
        n_patients=("subject_id", "nunique"),
    )
    .reset_index()
    .sort_values("n_rows", ascending=False)
)

label_breakdown

,label,n_rows,n_patients
3,unknown_no_followup_evidence,2139,1194
1,negative_observed,1604,860
2,positive,951,644
0,exclude_already_toxic,726,292


In [20]:
binary_label_breakdown = (
    binary_df.groupby(["label", "binary_label"], dropna=False)
    .agg(
        n_rows=("subject_id", "size"),
        n_patients=("subject_id", "nunique"),
    )
    .reset_index()
    .sort_values("n_rows", ascending=False)
)

binary_label_breakdown

,label,binary_label,n_rows,n_patients
0,negative_observed,0,1604,860
1,positive,1,951,644


In [21]:
drug_class_breakdown = (
    final_df.groupby("drug_classes_in_cycle", dropna=False)
    .agg(
        n_rows=("subject_id", "size"),
        n_patients=("subject_id", "nunique"),
        positives=("toxicity_in_window", "sum"),
    )
    .reset_index()
    .sort_values("n_rows", ascending=False)
)

drug_class_breakdown.head(25)

,drug_classes_in_cycle,n_rows,n_patients,positives
0,anthracycline,3003,1198,621
25,taxane,674,449,81
14,fluoropyrimidine,559,368,39
24,proteasome_inhibitor,441,275,94
27,tyrosine_kinase_inhibitor,201,117,39
22,immunomodulatory_agent,103,83,19
5,anthracycline | proteasome_inhibitor,84,31,6
28,vegf_inhibitor,80,65,3
16,fluoropyrimidine | taxane,52,35,11
18,her2_targeted,46,31,7


In [22]:
# Good sanity check: inspect patients with many cycle rows.
example_subjects = (
    final_df.groupby("subject_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .index
)

final_df[final_df["subject_id"].isin(example_subjects)].sort_values(["subject_id", "cycle_number"])

,subject_id,cycle_number,prediction_time,cycle_start_date,cycle_end_date,n_exposure_start_days_in_cycle,n_prescription_rows_in_cycle,drugs_in_cycle,drug_classes_in_cycle,toxicity_window_days,prediction_window_end,window_rationales,exposed_anthracycline,exposed_immune_checkpoint_inhibitor,exposed_her2_targeted,exposed_taxane,exposed_fluoropyrimidine,exposed_vegf_inhibitor,exposed_egfr_inhibitor,exposed_tyrosine_kinase_inhibitor,exposed_proteasome_inhibitor,exposed_immunomodulatory_agent,first_toxicity_time,first_toxicity_type,toxicity_in_window,pre_existing_cv_history,baseline_time,baseline_lvef,last_observation_time,observed_through_prediction_window,has_observation_in_prediction_window,eligible_for_binary_label,binary_label,label
557,11023442,1.0,2174-12-25 12:00:00,2174-12-25,2174-12-25,3,9.0,liposomal doxorubicin (doxil),anthracycline,365,2175-12-25 12:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,NaT,NaN,0,0,NaT,NaN,2176-03-09 08:00:00,1,1,1,0,negative_observed
558,11023442,2.0,2175-01-22 10:00:00,2175-01-22,2175-01-22,3,9.0,liposomal doxorubicin (doxil),anthracycline,365,2176-01-22 10:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,NaT,NaN,0,0,NaT,NaN,2176-03-09 08:00:00,1,1,1,0,negative_observed
559,11023442,3.0,2175-02-26 10:00:00,2175-02-26,2175-02-26,3,9.0,liposomal doxorubicin (doxil),anthracycline,365,2176-02-26 10:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,NaT,NaN,0,0,NaT,NaN,2176-03-09 08:00:00,1,1,1,0,negative_observed
560,11023442,4.0,2175-03-26 10:00:00,2175-03-26,2175-03-26,3,9.0,liposomal doxorubicin (doxil),anthracycline,365,2176-03-25 10:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,NaT,NaN,0,0,NaT,NaN,2176-03-09 08:00:00,0,1,0,<NA>,unknown_no_followup_evidence
561,11023442,5.0,2175-04-23 09:00:00,2175-04-23,2175-04-23,3,9.0,liposomal doxorubicin (doxil),anthracycline,365,2176-04-22 09:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,NaT,NaN,0,0,NaT,NaN,2176-03-09 08:00:00,0,1,0,<NA>,unknown_no_followup_evidence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4034,17424221,9.0,2167-03-17 14:00:00,2167-03-17,2167-03-17,1,1.0,doxorubicin,anthracycline,365,2168-03-16 14:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2166-08-21 15:00:00,lvef_ctrctd,0,0,2166-04-14 13:44:00,60.0,2167-08-19 18:00:00,0,1,0,<NA>,exclude_already_toxic
4035,17424221,10.0,2167-05-05 08:00:00,2167-05-05,2167-05-08,3,5.0,bortezomib | doxorubicin | pomalidomide,anthracycline | immunomodulatory_agent | proteasome_inhibitor,365,2168-05-04 08:00:00,Early-onset CTRCD/HF commonly assessed within 1 year | HF/ischemia/arrhythmia risk over months | Thrombotic/CV risk ...,1,0,0,0,0,0,0,0,1,1,2166-08-21 15:00:00,lvef_ctrctd,0,0,2166-04-14 13:44:00,60.0,2167-08-19 18:00:00,0,1,0,<NA>,exclude_already_toxic
4036,17424221,11.0,2167-06-02 17:00:00,2167-06-02,2167-06-02,2,4.0,doxorubicin | pomalidomide,anthracycline | immunomodulatory_agent,365,2168-06-01 17:00:00,Early-onset CTRCD/HF commonly assessed within 1 year | Thrombotic/CV risk over months,1,0,0,0,0,0,0,0,0,1,2166-08-21 15:00:00,lvef_ctrctd,0,0,2166-04-14 13:44:00,60.0,2167-08-19 18:00:00,0,1,0,<NA>,exclude_already_toxic
4037,17424221,12.0,2167-06-28 14:00:00,2167-06-28,2167-06-29,3,5.0,doxorubicin | pomalidomide,anthracycline | immunomodulatory_agent,365,2168-06-27 14:00:00,Early-onset CTRCD/HF commonly assessed within 1 year | Thrombotic/CV risk over months,1,0,0,0,0,0,0,0,0,1,2166-08-21 15:00:00,lvef_ctrctd,0,0,2166-04-14 13:44:00,60.0,2167-08-19 18:00:00,0,1,0,<NA>,exclude_already_toxic


## 9. Export modelling tables

In [23]:
write_dataframe(final_df, OUTPUT_DIR, "final_cycle_modeling_table")
write_dataframe(binary_df, OUTPUT_DIR, "final_cycle_binary_modeling_table")

label_breakdown.to_csv(OUTPUT_DIR / "label_breakdown.csv", index=False)
binary_label_breakdown.to_csv(OUTPUT_DIR / "binary_label_breakdown.csv", index=False)
drug_class_breakdown.to_csv(OUTPUT_DIR / "drug_class_breakdown.csv", index=False)

print("Wrote outputs to", OUTPUT_DIR)

wrote /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/data_outputs/cycle_modeling/final_cycle_modeling_table.csv
wrote /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/data_outputs/cycle_modeling/final_cycle_modeling_table.parquet
wrote /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/data_outputs/cycle_modeling/final_cycle_binary_modeling_table.csv
wrote /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/data_outputs/cycle_modeling/final_cycle_binary_modeling_table.parquet
Wrote outputs to /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/data_outputs/cycle_modeling


## 10. Optional patient-level split template

Only use this after you are ready to model. The split must be by `subject_id`, not by row/cycle.

In [ ]:
# Optional template:
# from sklearn.model_selection import train_test_split
#
# patient_ids = binary_df["subject_id"].drop_duplicates()
# train_ids, test_ids = train_test_split(patient_ids, test_size=0.2, random_state=42)
#
# train_df = binary_df[binary_df["subject_id"].isin(train_ids)].copy()
# test_df = binary_df[binary_df["subject_id"].isin(test_ids)].copy()
#
# print(train_df.shape, test_df.shape)
# print("Patient overlap:", set(train_df.subject_id) & set(test_df.subject_id))